In [ ]:
import numpy as np
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("jutrera/stanford-car-dataset-by-classes-folder")

print("Path to dataset files:", path)

100%|██████████| 1.83G/1.83G [00:21<00:00, 90.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jutrera/stanford-car-dataset-by-classes-folder/versions/2


In [ ]:
print(os.listdir(path))

['anno_test.csv', 'anno_train.csv', 'names.csv', 'car_data']


In [ ]:
train_dir=os.path.join(path,'car_data/car_data','train')
val_dir=os.path.join(path,'car_data/car_data','test')

In [ ]:
train_dir

'/root/.cache/kagglehub/datasets/jutrera/stanford-car-dataset-by-classes-folder/versions/2/car_data/car_data/train'

In [ ]:
#Image Augmentation
train_datagen=ImageDataGenerator(
    rescale=1./255
)
train_datagen

In [ ]:
val_datagen=ImageDataGenerator(
    rescale=1./255
)
val_datagen

In [ ]:
#image train and val data generate


train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical'
)
train_generator

Found 8144 images belonging to 196 classes.


In [ ]:
val_generator=val_datagen.flow_from_directory(
     val_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical'
)
val_generator

Found 8041 images belonging to 196 classes.


In [ ]:
#Build CNN Model
model=models.Sequential(
    [
        layers.Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3),padding='same'),
        layers.MaxPool2D((2,2)),

        layers.Conv2D(64,(3,3),activation='relu',padding='same'),
        layers.MaxPool2D((2,2)),

        layers.Conv2D(128,(3,3),activation='relu',padding='same'),
        layers.MaxPool2D((2,2)),

        layers.Flatten(),
        layers.Dense(128,activation='relu'),
        layers.Dense(196,activation='softmax')
    ]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 196)            │        25,284 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,312,964 (16.45 MB)

 Trainable params: 4,312,964 (16.45 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history=model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

Epoch 1/10
255/255 ━━━━━━━━━━━━━━━━━━━━ 64s 252ms/step - accuracy: 0.9861 - loss: 0.0512 - val_accuracy: 0.0144 - val_loss: 33.4120
Epoch 2/10
255/255 ━━━━━━━━━━━━━━━━━━━━ 63s 248ms/step - accuracy: 0.9801 - loss: 0.0859 - val_accuracy: 0.0142 - val_loss: 32.4238
Epoch 3/10
255/255 ━━━━━━━━━━━━━━━━━━━━ 63s 247ms/step - accuracy: 0.9802 - loss: 0.1067 - val_accuracy: 0.0139 - val_loss: 30.1976
Epoch 4/10
255/255 ━━━━━━━━━━━━━━━━━━━━ 63s 246ms/step - accuracy: 0.9786 - loss: 0.0898 - val_accuracy: 0.0142 - val_loss: 30.5944
Epoch 5/10
255/255 ━━━━━━━━━━━━━━━━━━━━ 73s 288ms/step - accuracy: 0.9856 - loss: 0.0622 - val_accuracy: 0.0159 - val_loss: 32.3912
Epoch 6/10
255/255 ━━━━━━━━━━━━━━━━━━━━ 63s 246ms/step - accuracy: 0.9897 - loss: 0.0425 - val_accuracy: 0.0149 - val_loss: 33.4831
Epoch 7/10
255/255 ━━━━━━━━━━━━━━━━━━━━ 63s 248ms/step - accuracy: 0.9934 - loss: 0.0285 - val_accuracy: 0.0170 - val_loss: 36.3955
Epoch 8/10
255/255 ━━━━━━━━━━━━━━━━━━━━ 63s 247ms/step - accuracy: 0.9942 - 

In [ ]:
# predict
from tensorflow.keras.preprocessing import image
img_path=os.path.join('/content/car_image.jpg')
img=image.load_img(img_path,target_size=(128,128))
img_array=image.img_to_array(img)
img_array=np.expand_dims(img_array,axis=0)
img_array=img_array/255.0
prediction=model.predict(img_array)
class_labels=train_generator.class_indices
predicted_class_index=np.argmax(prediction)
predicted_class_label=list(class_labels.keys())[list(class_labels.values()).index(predicted_class_index)]
predicted_class_label

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 694ms/step


'Chevrolet Malibu Sedan 2007'